<a href="https://colab.research.google.com/github/anujjakhotiya/AI-DL_2026/blob/main/Lab_09.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q datasets

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from datasets import load_dataset
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Load dataset
data = load_dataset("zh-plus/tiny-imagenet")
print(data)

train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(64, padding=4),
    transforms.ToTensor(),
    transforms.Normalize([0.4802,0.4481,0.3975],
                         [0.2770,0.2691,0.2821])
])

val_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.4802,0.4481,0.3975],
                         [0.2770,0.2691,0.2821])
])

class TinyDataset(Dataset):
    def __init__(self, data, tf):
        self.data, self.tf = data, tf

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        x = self.data[i]["image"].convert("RGB")
        y = self.data[i]["label"]
        return self.tf(x), y

train_ds = TinyDataset(data["train"], train_tf)
val_ds = TinyDataset(data["valid"], val_tf)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False, num_workers=2)

# Deep CNN
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),

            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128,128,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),

            nn.Conv2d(128,256,3,padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256,256,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),

            nn.Conv2d(256,512,3,padding=1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten(),
            nn.Linear(512*4*4,512), nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512,200)
        )

    def forward(self,x):
        return self.net(x)

model = CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

epochs = 5
tl, vl, ta, va = [], [], [], []

# Training
for e in range(epochs):
    model.train()
    loss_sum = correct = total = 0

    for x,y in train_loader:
        x,y = x.to(device),y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out,y)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item()*len(y)
        correct += (out.argmax(1)==y).sum().item()
        total += len(y)

    train_loss = loss_sum/total
    train_acc = 100*correct/total

    model.eval()
    loss_sum = correct = total = 0

    with torch.no_grad():
        for x,y in val_loader:
            x,y = x.to(device),y.to(device)
            out = model(x)
            loss_sum += criterion(out,y).item()*len(y)
            correct += (out.argmax(1)==y).sum().item()
            total += len(y)

    val_loss = loss_sum/total
    val_acc = 100*correct/total

    tl.append(train_loss); vl.append(val_loss)
    ta.append(train_acc); va.append(val_acc)

    print(f"Epoch {e+1}/{epochs} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

# Predictions
model.eval()
pred, true = [], []

with torch.no_grad():
    for x,y in val_loader:
        out = model(x.to(device))
        pred.extend(out.argmax(1).cpu().numpy())
        true.extend(y.numpy())

print("\nFinal Accuracy:", accuracy_score(true,pred)*100, "%")
print("\nClassification Report:\n",
      classification_report(true,pred,zero_division=0))

# Confusion Matrix
plt.figure(figsize=(8,7))
plt.imshow(confusion_matrix(true,pred))
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.colorbar()
plt.show()

# Loss
plt.plot(tl,label="Train")
plt.plot(vl,label="Validation")
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

# Accuracy
plt.plot(ta,label="Train")
plt.plot(va,label="Validation")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.legend()
plt.show()

# Sample predictions
x,y = next(iter(val_loader))
out = model(x.to(device))
p = out.argmax(1).cpu()

plt.figure(figsize=(10,6))
for i in range(8):
    plt.subplot(2,4,i+1)
    img = x[i]*torch.tensor([0.2770,0.2691,0.2821]).view(3,1,1)
    img += torch.tensor([0.4802,0.4481,0.3975]).view(3,1,1)
    plt.imshow(torch.clamp(img,0,1).permute(1,2,0))
    plt.title(f"True:{y[i]} Pred:{p[i]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

torch.save(model.state_dict(), "tiny_imagenet_cnn.pth")

print("\nConclusion: The deeper CNN learned image features through multiple convolutional layers. "
      "Training and validation curves show model convergence, while dropout and weight decay "
      "help reduce overfitting.")

Device: cuda
DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 100000
    })
    valid: Dataset({
        features: ['image', 'label'],
        num_rows: 10000
    })
})
